# Inferência de Nome do Pai e Nome da Mãe a partir de Relação Familiar (Didático)

Este notebook reorganiza o seu script de inferência de parentesco para ficar **mais didático e reproduzível**.

## Objetivo

A partir de um Parquet (gerado na etapa anterior do seu pipeline), inferir para cada morador:

- `nome_mae` (string)
- `nome_pai` (string)

e salvar um **Parquet de saída** pronto para dar **merge** com o Parquet anterior.

---

## Entradas esperadas (no Parquet da etapa anterior)

O Parquet de entrada precisa conter, no mínimo, estas colunas:

**Chaves para merge**
- `ID_DOMICILIO` (id do domicílio / família)
- `ID_MORADOR` (id do morador / pessoa)

**Nome**
- `PECP0029` (nome)
- `PECP0357` (sobrenome)

**Relação familiar e sexo**
- `PECP0004` (código de relação familiar)
- `PECP0002` (código de sexo)

> Se algum nome de coluna estiver diferente no seu Parquet real, ajuste a lista `USE_COLS` no início.


## Saída final (o que você vai obter)

### Arquivo Parquet: `pais_inferidos.parquet`

Esse arquivo conterá **somente** as colunas necessárias para fazer merge com o Parquet anterior, sem duplicar o dataset inteiro:

- `ID_DOMICILIO` (string)
- `ID_MORADOR` (string)
- `nome_mae` (string, pode ser nulo)
- `nome_pai` (string, pode ser nulo)

Opcionalmente, também incluímos:
- `nome_completo` (string) — ajuda em auditoria/QA

### Como fazer o merge depois

No Pandas:

```python
base = pd.read_parquet(PARQUET_ENTRADA)
pais = pd.read_parquet("pais_inferidos.parquet")

base2 = base.merge(pais[["ID_DOMICILIO","ID_MORADOR","nome_mae","nome_pai"]],
                   on=["ID_DOMICILIO","ID_MORADOR"],
                   how="left")
```

Isso adiciona as colunas de pai/mãe mantendo **todas as linhas** da base original.


---

# 0) Setup
Ajuste o caminho do Parquet de entrada (gerado na etapa anterior).


In [ ]:
from pathlib import Path
import pandas as pd

# ✅ AJUSTE AQUI:
PARQUET_ENTRADA = Path("/path/to/join_final.parquet")

# Saídas
OUT_PARQUET = Path("pais_inferidos.parquet")
OUT_CSV = Path("pais_inferidos.csv")

# Colunas mínimas para inferência
USE_COLS = [
    "ID_DOMICILIO",
    "ID_MORADOR",
    "PECP0029",
    "PECP0357",
    "PECP0004",
    "PECP0002",
]


# 1) Carregar dados do Parquet (apenas colunas necessárias)

Como o Parquet é colunar, ler apenas `USE_COLS` é rápido e economiza memória.


In [ ]:
df = pd.read_parquet(PARQUET_ENTRADA, columns=USE_COLS)

print("Linhas:", len(df))
display(df.head(3))


# 2) Preparação e normalização

Criamos:
- `nome_completo`
- `relacao` (código de relação, como string, zero-padded)
- `sexo` (código de sexo, como string)

> Se seus códigos forem diferentes, ajuste as constantes no próximo bloco.


In [ ]:
df["ID_DOMICILIO"] = df["ID_DOMICILIO"].astype(str)
df["ID_MORADOR"] = df["ID_MORADOR"].astype(str)

df["PECP0029"] = df["PECP0029"].fillna("").astype(str)
df["PECP0357"] = df["PECP0357"].fillna("").astype(str)

df["nome_completo"] = (df["PECP0029"].str.strip() + " " + df["PECP0357"].str.strip()).str.strip()

df["relacao"] = df["PECP0004"].astype(str).str.strip().str.zfill(2)
df["sexo"] = df["PECP0002"].astype(str).str.strip()

display(df[["ID_DOMICILIO","ID_MORADOR","nome_completo","relacao","sexo"]].head(5))


# 3) Regras de inferência (didáticas)

Mantemos a ideia do seu script: inferir pai/mãe por domicílio a partir dos papéis:

- Responsável (`01`)
- Cônjuge/companheiro (`02`, `03`)
- Filho de ambos (`04`)
- Filho só do responsável (`05`)
- Enteado (`06`)
- Pais do cônjuge (sogro/sogra) (`09`)
- Outros pais (`08`)

**Convenção de sexo** usada (ajuste se necessário):
- `1` = masculino
- `2` = feminino


In [ ]:
RESPONSAVEL = "01"
CONJUGE = {"02", "03"}
FILHO_AMBOS = "04"
FILHO_RESP = "05"
ENTEADO = "06"
OUTROS_PAIS = "08"
SOGROS = "09"

SEXO_MASC = "1"
SEXO_FEM = "2"

def primeiro_nome_por_sexo(pessoas, sexo_alvo):
    """Retorna o primeiro nome (nome_completo) na lista 'pessoas' com sexo==sexo_alvo."""
    for p in pessoas:
        if str(p.get("sexo","")).strip() == str(sexo_alvo):
            return p.get("nome_completo")
    return None

def inferir_pais_familia(membros):
    """Aplica as regras e devolve lista com (ID_DOMICILIO, ID_MORADOR, nome_mae, nome_pai)."""

    responsavel = [m for m in membros if m["relacao"] == RESPONSAVEL]
    conjuges = [m for m in membros if m["relacao"] in CONJUGE]
    sogros = [m for m in membros if m["relacao"] == SOGROS]
    outros_pais = [m for m in membros if m["relacao"] == OUTROS_PAIS]

    # Candidatos gerais de pais: responsável + cônjuge + outros pais
    nome_mae_casa = primeiro_nome_por_sexo(responsavel + conjuges + outros_pais, SEXO_FEM)
    nome_pai_casa = primeiro_nome_por_sexo(responsavel + conjuges + outros_pais, SEXO_MASC)

    # Mapeamento de sogros para o cônjuge (quando existe)
    sogros_por_conjuge = {}
    if conjuges:
        id_conjuge = conjuges[0]["ID_MORADOR"]
        for s in sogros:
            if s["sexo"] == SEXO_FEM:
                sogros_por_conjuge.setdefault(id_conjuge, {})["nome_mae"] = s["nome_completo"]
            elif s["sexo"] == SEXO_MASC:
                sogros_por_conjuge.setdefault(id_conjuge, {})["nome_pai"] = s["nome_completo"]

    saida = []
    for m in membros:
        rel = m["relacao"]
        pid = m["ID_MORADOR"]

        nome_mae = None
        nome_pai = None

        if rel == FILHO_AMBOS:
            nome_mae, nome_pai = nome_mae_casa, nome_pai_casa

        elif rel == FILHO_RESP:
            # usa apenas o responsável como pai/mãe
            nome_mae = primeiro_nome_por_sexo(responsavel, SEXO_FEM)
            nome_pai = primeiro_nome_por_sexo(responsavel, SEXO_MASC)

        elif rel == ENTEADO:
            # usa apenas o cônjuge como pai/mãe
            nome_mae = primeiro_nome_por_sexo(conjuges, SEXO_FEM)
            nome_pai = primeiro_nome_por_sexo(conjuges, SEXO_MASC)

        elif rel in CONJUGE:
            # tenta usar sogros como pais do cônjuge
            info = sogros_por_conjuge.get(pid, {})
            nome_mae = info.get("nome_mae")
            nome_pai = info.get("nome_pai")

        saida.append({
            "ID_DOMICILIO": m["ID_DOMICILIO"],
            "ID_MORADOR": pid,
            "nome_completo": m["nome_completo"],
            "nome_mae": nome_mae,
            "nome_pai": nome_pai,
        })

    return saida


# 4) Execução (Pandas) — agrupando por domicílio

Se sua base é grande e você tiver RAM suficiente, este método é simples e eficiente.


In [ ]:
work = df[["ID_DOMICILIO","ID_MORADOR","nome_completo","relacao","sexo"]].copy()
work = work.sort_values(["ID_DOMICILIO","ID_MORADOR"])

out_rows = []
for id_dom, g in work.groupby("ID_DOMICILIO", sort=False):
    membros = []
    for _, r in g.iterrows():
        membros.append({
            "ID_DOMICILIO": r["ID_DOMICILIO"],
            "ID_MORADOR": r["ID_MORADOR"],
            "nome_completo": r["nome_completo"],
            "relacao": r["relacao"],
            "sexo": r["sexo"],
        })
    out_rows.extend(inferir_pais_familia(membros))

pais = pd.DataFrame(out_rows)
display(pais.head(10))


# 5) Garantir chaves e unicidade para merge

Para dar merge com segurança, o dataset de saída deve ter:
- `ID_DOMICILIO` + `ID_MORADOR` como chaves
- no máximo 1 linha por chave

Aqui vamos:
- converter chaves para string
- checar duplicidade
- (se houver duplicidade) manter a primeira ocorrência


In [ ]:
pais["ID_DOMICILIO"] = pais["ID_DOMICILIO"].astype(str)
pais["ID_MORADOR"] = pais["ID_MORADOR"].astype(str)

dups = pais.duplicated(["ID_DOMICILIO","ID_MORADOR"]).sum()
print("Duplicidades (ID_DOMICILIO, ID_MORADOR):", dups)

if dups > 0:
    pais = pais.drop_duplicates(["ID_DOMICILIO","ID_MORADOR"], keep="first")
    print("Após drop_duplicates:", len(pais))

# Saída mínima para merge
pais_merge = pais[["ID_DOMICILIO","ID_MORADOR","nome_mae","nome_pai"]].copy()
display(pais_merge.head(5))


# 6) Salvar saída (Parquet e CSV)

- `pais_inferidos.parquet` é o arquivo ideal para merge (colunas mínimas).
- O CSV é opcional, útil para inspeção manual.


In [ ]:
pais_merge.to_parquet(OUT_PARQUET, index=False)
pais_merge.to_csv(OUT_CSV, index=False, encoding="utf-8")

print("Salvo Parquet:", OUT_PARQUET.resolve())
print("Salvo CSV:", OUT_CSV.resolve())


# 7) Exemplo: fazer merge com o Parquet anterior

⚠️ Este exemplo lê o Parquet de entrada completo, o que pode ser grande.
Se quiser, você pode ler só as colunas necessárias do Parquet de entrada.


In [ ]:
# Exemplo de merge
base = pd.read_parquet(PARQUET_ENTRADA)  # ou columns=[...]
base2 = base.merge(pais_merge, on=["ID_DOMICILIO","ID_MORADOR"], how="left")

print("Base original:", base.shape)
print("Base com pais:", base2.shape)
display(base2[["ID_DOMICILIO","ID_MORADOR","nome_mae","nome_pai"]].head(10))


---

# Opcional: modo streaming (DuckDB) para bases gigantes

Se você preferir não carregar tudo em RAM, dá para iterar em batches ordenados por domicílio
e processar uma família por vez, mantendo memória baixa.

> Mantive o esqueleto no fim do notebook; você só precisa ativar e ajustar o caminho.


In [ ]:
import duckdb

def inferir_pais_streaming(parquet_path: str, out_parquet: str):
    con = duckdb.connect()
    q = f'''
    SELECT
      CAST(ID_DOMICILIO AS VARCHAR) AS ID_DOMICILIO,
      CAST(ID_MORADOR AS VARCHAR) AS ID_MORADOR,
      COALESCE(CAST(PECP0029 AS VARCHAR), '') AS PECP0029,
      COALESCE(CAST(PECP0357 AS VARCHAR), '') AS PECP0357,
      CAST(PECP0004 AS VARCHAR) AS PECP0004,
      CAST(PECP0002 AS VARCHAR) AS PECP0002
    FROM read_parquet('{parquet_path}')
    ORDER BY 1, 2
    '''
    cur = con.execute(q)

    import pyarrow as pa
    import pyarrow.parquet as pq

    batches = []
    current_fam = None
    membros = []

    def flush_family(membros_local):
        if not membros_local:
            return []
        return inferir_pais_familia(membros_local)

    while True:
        rows = cur.fetchmany(100_000)
        if not rows:
            break

        for ID_DOMICILIO, ID_MORADOR, n1, n2, rel, sexo in rows:
            fam = ID_DOMICILIO
            if current_fam is None:
                current_fam = fam

            if fam != current_fam:
                res = flush_family(membros)
                if res:
                    df_res = pd.DataFrame(res)[["ID_DOMICILIO","ID_MORADOR","nome_mae","nome_pai"]]
                    batches.append(pa.Table.from_pandas(df_res, preserve_index=False))
                membros = []
                current_fam = fam

            membros.append({
                "ID_DOMICILIO": str(ID_DOMICILIO),
                "ID_MORADOR": str(ID_MORADOR),
                "nome_completo": (str(n1).strip() + " " + str(n2).strip()).strip(),
                "relacao": str(rel).strip().zfill(2),
                "sexo": str(sexo).strip(),
            })

    # última família
    res = flush_family(membros)
    if res:
        df_res = pd.DataFrame(res)[["ID_DOMICILIO","ID_MORADOR","nome_mae","nome_pai"]]
        batches.append(pa.Table.from_pandas(df_res, preserve_index=False))

    if not batches:
        raise ValueError("Nenhum resultado gerado.")

    table = pa.concat_tables(batches)
    pq.write_table(table, out_parquet)

    return out_parquet

# Exemplo:
# inferir_pais_streaming(str(PARQUET_ENTRADA), "pais_inferidos_streaming.parquet")
